In [1]:
import os
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


project_root = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / 'pyproject.toml').is_file()
)
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from config_plan import mfcc_sliding_window_data_pipeline_config
from core.data_pipeline import DataPipeline


config = mfcc_sliding_window_data_pipeline_config
pipeline = DataPipeline.create(config)
source_series = pipeline.source_reader.get_source_series()

print(f'Loaded {len(source_series)} eligible source recordings.')
print(f'Window size: {config.segmenter.window_size} samples')
print(f'Stride: {config.segmenter.stride} samples')
print(f'Overlap threshold: {config.segmenter.overlap_threshold:.0%}')

/home/user/Hehe/allOfMyCode/workspace_1/CoughClassificationProject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 578 eligible source recordings.
Window size: 8200 samples
Stride: 4100 samples
Overlap threshold: 70%


# Sliding-window pipeline audit

This notebook visually audits one source recording through the configured pipeline:

1. waveform and detected cough intervals;
2. fixed-width windows, their maximum cough overlap, and their final label;
3. MFCCs for representative windows.

It uses `mfcc_sliding_window_data_pipeline_config`, so the audit always reflects the current experiment settings.

In [2]:
def source_summary(series):
    cough_segments = series.metadata.get('detected_cough_segments', [])
    return {
        'patient_id': series.metadata.get('patient_id'),
        'audio': str(series.metadata.get('cough_audio', '')),
        'recording_label': series.label,
        'duration_seconds': len(series.value) / series.metadata['sample_rate'],
        'detected_coughs': len(cough_segments),
    }

recordings = pd.DataFrame(source_summary(series) for series in source_series)
recordings.sort_values(['recording_label', 'detected_coughs'], ascending=[False, False]).head(20)

,patient_id,audio,recording_label,duration_seconds,detected_coughs
300,11076,s3:/elderly-care-audio/audio/11076/cough-2025-...,1,20.178125,7
111,11092,s3:/elderly-care-audio/audio/ 11092/cough-2026...,1,22.082187,6
282,11077,s3:/elderly-care-audio/audio/11077/cough-2025-...,1,13.258625,6
345,11045,s3:/elderly-care-audio/audio/11045/cough-2025-...,1,21.888000,6
369,11026,s3:/elderly-care-audio/audio/11026/cough-2025-...,1,13.184000,6
41,1134,s3:/elderly-care-audio/audio/1134/cough-2026-0...,1,9.334437,5
280,11078,s3:/elderly-care-audio/audio/11078/cough-2025-...,1,19.202938,5
320,11062,s3:/elderly-care-audio/audio/11062/cough-2025-...,1,23.424000,5
325,11061,s3:/elderly-care-audio/audio/11061/cough-2025-...,1,21.034688,5
338,11053,s3:/elderly-care-audio/audio/11053/cough-2025-...,1,15.786687,5


Choose a row from the table above. The default selects the first recording with detected cough intervals; change `RECORDING_INDEX` to inspect a specific recording.

In [3]:
RECORDING_INDEX = int(recordings.query('detected_coughs > 0').index[0])
series = source_series[RECORDING_INDEX]

print(source_summary(series))

{'patient_id': '011', 'audio': 's3:/elderly-care-audio/audio/011/cough-2026-03-09T06:13:58.767851Z.wav', 'recording_label': 0, 'duration_seconds': 6.5480625, 'detected_coughs': 2}


In [4]:
def maximum_overlap_ratio(window_start, window_end, cough_segments):
    window_size = window_end - window_start
    if not cough_segments:
        return 0.0

    return max(
        max(0, min(window_end, cough_end + 1) - max(window_start, cough_start)) / window_size
        for cough_start, cough_end in cough_segments
    )


def describe_windows(series, segmenter):
    cough_segments = series.metadata.get('detected_cough_segments', [])
    rows = []

    for start in range(0, len(series.value), segmenter.stride):
        end = start + segmenter.window_size
        if end > len(series.value) and not segmenter.keep_short_segments:
            continue

        overlap = maximum_overlap_ratio(start, end, cough_segments)
        label = segmenter._assign_label(start, end, cough_segments, series.label)
        rows.append({
            'start': start,
            'end': min(end, len(series.value)),
            'start_seconds': start / series.metadata['sample_rate'],
            'end_seconds': min(end, len(series.value)) / series.metadata['sample_rate'],
            'max_overlap': overlap,
            'label': label,
        })

    return pd.DataFrame(rows)


windows = describe_windows(series, config.segmenter)
display(windows)
print('Window labels:', dict(Counter(windows['label'])))

,start,end,start_seconds,end_seconds,max_overlap,label
0,0,8200,0.00000,0.51250,0.250732,0
1,4100,12300,0.25625,0.76875,0.750732,0
2,8200,16400,0.51250,1.02500,1.000000,0
3,12300,20500,0.76875,1.28125,1.000000,0
4,16400,24600,1.02500,1.53750,1.000000,0
5,20500,28700,1.28125,1.79375,1.000000,0
6,24600,32800,1.53750,2.05000,1.000000,0
7,28700,36900,1.79375,2.30625,1.000000,0
8,32800,41000,2.05000,2.56250,1.000000,0
9,36900,45100,2.30625,2.81875,1.000000,0


Window labels: {0: 24}


In [5]:
def plot_window_audit(series, windows):
    sample_rate = series.metadata['sample_rate']
    times = np.arange(len(series.value)) / sample_rate
    cough_segments = series.metadata.get('detected_cough_segments', [])

    figure, (waveform_axis, windows_axis) = plt.subplots(
        nrows=2, figsize=(16, 7), sharex=True, height_ratios=[3, 1], layout='constrained'
    )
    waveform_axis.plot(times, series.value, color='midnightblue', linewidth=0.4)
    waveform_axis.set(title='Waveform with detected cough intervals', ylabel='Amplitude')

    for cough_start, cough_end in cough_segments:
        waveform_axis.axvspan(cough_start / sample_rate, (cough_end + 1) / sample_rate,
                             color='tab:green', alpha=0.3)

    colors = {0: 'tab:blue', 1: 'tab:orange'}
    for index, window in windows.iterrows():
        width = window.end_seconds - window.start_seconds
        windows_axis.barh(0, width, left=window.start_seconds, height=0.5,
                          color=colors.get(window.label, 'gray'), alpha=0.75)
        windows_axis.text(window.start_seconds + width / 2, 0,
                          f"#{index}\n{window.max_overlap:.0%}",
                          ha='center', va='center', fontsize=8)

    windows_axis.set(
        xlabel='Time (seconds)', yticks=[],
        title='Windows: blue = label 0, orange = label 1; text = maximum cough overlap'
    )
    return figure


plot_window_audit(series, windows);

In [6]:
# Select a label-0, a label-1, and a threshold-near window when available.
candidates = pd.concat([
    windows.groupby('label', group_keys=False).head(1),
    windows.assign(distance=(windows.max_overlap - config.segmenter.overlap_threshold).abs())
           .nsmallest(1, 'distance').drop(columns='distance'),
]).drop_duplicates(subset='start')

raw_windows = config.segmenter.segment([series])
selected_examples = [raw_windows[index] for index in candidates.index]
mfcc_examples = config.transformer.transform(selected_examples)

figure, axes = plt.subplots(1, len(mfcc_examples), figsize=(6 * len(mfcc_examples), 4), squeeze=False, layout='constrained')
for axis, example, (_, window) in zip(axes[0], mfcc_examples, candidates.iterrows()):
    image = axis.imshow(example.value.T, aspect='auto', origin='lower', cmap='magma')
    axis.set(
        title=f"Label {window.label}; overlap {window.max_overlap:.1%}",
        xlabel='MFCC frame', ylabel='MFCC coefficient',
    )
    figure.colorbar(image, ax=axis, label='MFCC value')

plt.show()

/tmp/ipykernel_253304/559752117.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What to inspect

- Does every orange window have the intended amount of detected cough signal?
- Are blue windows near a cough boundary behaving as expected at the configured threshold?
- Do the MFCCs visibly distinguish cough-containing windows from mostly background windows?

Repeat the notebook with several recordings, especially mixed-label and threshold-near cases, before committing to a training run.